In [ ]:
%load_ext autoreload
%autoreload 3
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import sys, warnings
import numpy as np

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from config import SPE1_PICKLE_ROOT
from mlm_pop_ridge_utils import (
    build_meta_df,
    run_all_meta_regressions,
    plot_population_effects,
    plot_metadata_effects,
    plot_credible_pairs,
    META_NAMES, META_LABELS,
)
from pop_ridge_utils import load_population_results, aggregate_population
from ridge_regression_utils import FEAT_LABELS

warnings.filterwarnings('ignore')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
RIDGE_PICKLE_DIR = SPE1_PICKLE_ROOT + '/ridge_regression_pickles'
MLM_SAVE_PATH    = RIDGE_PICKLE_DIR + '/mlm_meta_regression_results.pkl'

CI              = 0.94   # confidence interval width
FDR_Q           = 0.05
FORCE_RECOMPUTE = True   # set False after first run

In [ ]:
# ── Load population results ────────────────────────────────────────────────────
all_results, cell_ids, target_names, predictor_sets = load_population_results(RIDGE_PICKLE_DIR)
_, _, beta_pop = aggregate_population(all_results, cell_ids, target_names, predictor_sets)

feat_labels   = FEAT_LABELS
target_labels = (
    [f'Pre {l}'     for l in feat_labels] +
    [f'Pre−BL {l}'  for l in feat_labels] +
    [f'Post {l}'    for l in feat_labels] +
    [f'Post−BL {l}' for l in feat_labels] +
    [f'Δ {l}'       for l in feat_labels]
)

print(f'{len(cell_ids)} cells  |  {len(target_names)} targets')

In [ ]:
# ── Build metadata matrix ──────────────────────────────────────────────────────
meta_df = build_meta_df(cell_ids)
print(meta_df.describe())
meta_df.head()

In [ ]:
# ── Run MLM meta-regression (OLS + HC3 robust SEs + FDR) ─────────────────────
# 200 models (25 targets × 8 features). Runs in seconds.
# FDR applied across all 200 intercept p-values, and separately per metadata variable.
df_mlm = run_all_meta_regressions(
    beta_pop, meta_df, target_names, target_labels,
    ci=CI, fdr_q=FDR_Q,
    save_path=MLM_SAVE_PATH, force_recompute=FORCE_RECOMPUTE,
)
print(df_mlm.shape)
df_mlm.head()

In [ ]:
# ── Population intercept forest plot ──────────────────────────────────────────
# μ + 94% CI for every (target × feature) pair.
# Filled diamond = FDR-significant. Red = positive, Blue = negative.
plot_population_effects(df_mlm, target_labels)

In [ ]:
# ── FDR-significant population effects ───────────────────────────────────────
plot_credible_pairs(df_mlm)

In [ ]:
# ── Metadata effects heatmap ──────────────────────────────────────────────────
# γ for each metadata variable across (target × feature) pairs. ★ = FDR-significant.
plot_metadata_effects(df_mlm, target_labels)